# Resident Transfer Risk Pipeline

**Primary question:** Which residents are at risk of being **Transferred** instead of **Closed**?

**Action question:** What can we do to support those at-risk residents?

This notebook is structured to satisfy sections **2-6**:
1. Data Acquisition, Preparation & Exploration
2. Modeling & Feature Selection
3. Evaluation & Interpretation
4. Causal and Relationship Analysis
5. Deployment Notes

In [ ]:
# Setup: imports, reproducibility, and paths
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'datasets'
ARTIFACT_DIR = Path.cwd() / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Data dir:', DATA_DIR)
print('Artifact dir:', ARTIFACT_DIR)

## 2) Data Acquisition, Preparation & Exploration

### Data sources and join logic
- `residents.csv`: final outcome (`case_status`) and case context.
- `incident_reports.csv`: incident burden, severity, and unresolved incidents.
- `education_records.csv`: school engagement and progress trends.

**Join key:** `resident_id`.

**Prediction framing choice:** We model only residents with final outcomes (`Closed`, `Transferred`).
- Target `y = 1` if `Transferred`, `0` if `Closed`.
- This keeps the question aligned to outcome comparison and avoids mixing ongoing (`Active`) cases with unknown future outcomes.

In [ ]:
# Load datasets
res = pd.read_csv(DATA_DIR / 'residents.csv')
inc = pd.read_csv(DATA_DIR / 'incident_reports.csv')
edu = pd.read_csv(DATA_DIR / 'education_records.csv')

print('residents:', res.shape)
print('incident_reports:', inc.shape)
print('education_records:', edu.shape)

for c in ['date_of_admission', 'date_enrolled', 'date_closed', 'created_at']:
    if c in res.columns:
        res[c] = pd.to_datetime(res[c], errors='coerce')

for c in ['incident_date', 'resolution_date']:
    if c in inc.columns:
        inc[c] = pd.to_datetime(inc[c], errors='coerce')

if 'record_date' in edu.columns:
    edu['record_date'] = pd.to_datetime(edu['record_date'], errors='coerce')

In [ ]:
# Predictive feature engineering: only information available by day-30 after enrollment
PREDICTION_WINDOW_DAYS = 30
severity_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}

# Base residents filtered to final outcomes
base = res[res['case_status'].isin(['Closed', 'Transferred'])].copy()
base['target_transferred'] = (base['case_status'] == 'Transferred').astype(int)
base['prediction_cutoff_date'] = base['date_enrolled'] + pd.to_timedelta(PREDICTION_WINDOW_DAYS, unit='D')

# Build incident features restricted to first 30 days post enrollment
inc2 = inc.copy()
inc2['severity_num'] = inc2['severity'].map(severity_map)
inc2['resolved_num'] = inc2['resolved'].astype(str).str.lower().map({'true': 1, 'false': 0})
inc2['follow_up_num'] = inc2['follow_up_required'].astype(str).str.lower().map({'true': 1, 'false': 0})

inc30 = base[['resident_id', 'date_enrolled', 'prediction_cutoff_date']].merge(
    inc2, on='resident_id', how='left'
)
inc30 = inc30[
    inc30['incident_date'].notna()
    & inc30['date_enrolled'].notna()
    & (inc30['incident_date'] >= inc30['date_enrolled'])
    & (inc30['incident_date'] <= inc30['prediction_cutoff_date'])
].copy()
inc30['unresolved_high'] = ((inc30['resolved_num'] == 0) & (inc30['severity_num'] >= 3)).astype(int)

inc_agg = inc30.groupby('resident_id', as_index=False).agg(
    incident_count_30d=('incident_id', 'count'),
    incident_severity_mean_30d=('severity_num', 'mean'),
    incident_severity_max_30d=('severity_num', 'max'),
    unresolved_ratio_30d=('resolved_num', lambda s: 1 - s.fillna(0).mean()),
    unresolved_high_count_30d=('unresolved_high', 'sum'),
    follow_up_ratio_30d=('follow_up_num', 'mean'),
)

# Build education features restricted to first 30 days post enrollment
edu30 = base[['resident_id', 'date_enrolled', 'prediction_cutoff_date']].merge(
    edu, on='resident_id', how='left'
)
edu30 = edu30[
    edu30['record_date'].notna()
    & edu30['date_enrolled'].notna()
    & (edu30['record_date'] >= edu30['date_enrolled'])
    & (edu30['record_date'] <= edu30['prediction_cutoff_date'])
].copy()

edu_agg = edu30.sort_values(['resident_id', 'record_date']).groupby('resident_id', as_index=False).agg(
    edu_records_30d=('education_record_id', 'count'),
    attendance_mean_30d=('attendance_rate', 'mean'),
    attendance_last_30d=('attendance_rate', 'last'),
    progress_mean_30d=('progress_percent', 'mean'),
    progress_last_30d=('progress_percent', 'last'),
)
edu_agg['progress_delta_30d'] = edu_agg['progress_last_30d'] - edu_agg['progress_mean_30d']
edu_agg['attendance_delta_30d'] = edu_agg['attendance_last_30d'] - edu_agg['attendance_mean_30d']

# Merge logic: left joins keep the resident outcome cohort and attach early-window signals
joined = (
    base
    .merge(inc_agg, on='resident_id', how='left')
    .merge(edu_agg, on='resident_id', how='left')
)

print('Modeling population shape:', joined.shape)
print('Transferred rate:', round(joined['target_transferred'].mean(), 3))
print('Residents with 30-day incident signals:', joined['incident_count_30d'].notna().sum())
print('Residents with 30-day education signals:', joined['edu_records_30d'].notna().sum())

In [ ]:
# Exploratory checks: missingness, distributions, correlation snapshot
missing = joined.isna().mean().sort_values(ascending=False).head(20)
print('Top missingness (%):')
display((missing * 100).round(1).to_frame('missing_pct'))

print('\nOutcome counts:')
print(joined['case_status'].value_counts())

num_cols_preview = [
    'incident_count', 'incident_severity_mean', 'unresolved_ratio',
    'attendance_mean', 'progress_mean', 'length_of_stay'
]
for c in num_cols_preview:
    if c in joined.columns:
        print(f"{c}:\n", joined[c].describe(), "\n")

corr_cols = [c for c in ['target_transferred','incident_count','incident_severity_mean','unresolved_ratio','attendance_mean','progress_mean','progress_delta'] if c in joined.columns]
if len(corr_cols) >= 3:
    corr = joined[corr_cols].corr(numeric_only=True)
    print('Correlation matrix (numeric subset):')
    display(corr)

# Simple outlier view for incident_count
if 'incident_count' in joined.columns:
    plt.figure(figsize=(6, 3))
    plt.boxplot(joined['incident_count'].fillna(0))
    plt.title('Incident count outlier check')
    plt.show()

### Feature engineering decisions
- Keep resident-level static context known early in the case lifecycle.
- Add cross-table behavior features from incidents in the first 30 days.
- Add education momentum features from records in the first 30 days.
- Exclude direct identifiers (`resident_id`, case codes) to prevent memorization.
- Exclude leakage fields unavailable at prediction time (`date_closed`, `days_enrolled_to_closed`, post-outcome statuses).

This is implemented as a reproducible sklearn `Pipeline` with `ColumnTransformer`, so transformations are identical in train/test and future scoring.

In [ ]:
# Build model-ready matrix with leakage audit and time-based split
leakage_cols = [
    'target_transferred', 'case_status', 'resident_id', 'case_control_no', 'internal_code',
    'date_closed', 'days_enrolled_to_closed', 'notes_restricted', 'created_at',
    'reintegration_status'
]

leakage_audit = pd.DataFrame([
    {'feature': 'date_closed', 'keep_for_prediction': False, 'reason': 'Known only at closure/transfer outcome'},
    {'feature': 'days_enrolled_to_closed', 'keep_for_prediction': False, 'reason': 'Direct outcome timing leakage'},
    {'feature': 'reintegration_status', 'keep_for_prediction': False, 'reason': 'Can be updated late in case lifecycle'},
    {'feature': 'incident_count_30d', 'keep_for_prediction': True, 'reason': 'Observable by day-30 prediction cutoff'},
    {'feature': 'progress_delta_30d', 'keep_for_prediction': True, 'reason': 'Built only from early education records'},
])
display(leakage_audit)

X = joined.drop(columns=[c for c in leakage_cols if c in joined.columns], errors='ignore')
y = joined['target_transferred']

# Time anchor for forward split
if 'date_enrolled' in joined.columns:
    split_time = joined['date_enrolled'].quantile(0.75)
    train_mask = joined['date_enrolled'] <= split_time
    test_mask = joined['date_enrolled'] > split_time
else:
    # Fallback if date_enrolled is unavailable
    train_mask = pd.Series([True] * len(joined))
    test_mask = pd.Series([False] * len(joined))

# Keep date columns only as engineered durations if needed; drop raw datetimes
for c in X.columns:
    if str(X[c].dtype).startswith('datetime64'):
        X = X.drop(columns=[c])

numeric_features = X.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

# Safety fallback if temporal split is too small/imbalanced
if len(X_test) < 5 or y_test.nunique() < 2 or y_train.nunique() < 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=SEED, stratify=y
    )
    split_note = 'Fallback to stratified random split (insufficient temporal holdout).'
else:
    split_note = f'Temporal split at enrollment date <= {split_time.date()} for train, later for test.'

print(split_note)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Transferred rate (train/test):', round(y_train.mean(), 3), round(y_test.mean(), 3))

In [ ]:
## 3) Modeling & Feature Selection

models = {
    'logistic': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
    'random_forest': RandomForestClassifier(n_estimators=400, random_state=SEED, class_weight='balanced'),
    'gradient_boosting': GradientBoostingClassifier(random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
comparison = []
trained = {}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', clf)])
    auc_pr = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='average_precision').mean()
    f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1').mean()
    comparison.append({'model': name, 'cv_avg_precision': auc_pr, 'cv_f1': f1})
    pipe.fit(X_train, y_train)
    trained[name] = pipe

comparison_df = pd.DataFrame(comparison).sort_values('cv_avg_precision', ascending=False)
display(comparison_df)

best_name = comparison_df.iloc[0]['model']
best_model = trained[best_name]
print('Selected model:', best_name)

# Hyperparameter tuning for random forest (if selected)
if best_name == 'random_forest':
    param_grid = {
        'model__n_estimators': [200, 400, 700],
        'model__max_depth': [None, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
    }
    grid = GridSearchCV(
        best_model,
        param_grid=param_grid,
        scoring='average_precision',
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    print('Best RF params:', grid.best_params_)

trained['selected'] = best_model

In [ ]:
# Feature selection / importance view
selected_model = trained['selected']

proba = selected_model.predict_proba(X_test)[:, 1]
pred_default = (proba >= 0.5).astype(int)

print('Default threshold classification report:')
print(classification_report(y_test, pred_default, digits=3))

# Business-weighted threshold: favor recall for transferred residents
prec, rec, thr = precision_recall_curve(y_test, proba)
valid_idx = np.where(rec[:-1] >= 0.75)[0]
threshold = float(thr[valid_idx[0]]) if len(valid_idx) else 0.5
pred_operational = (proba >= threshold).astype(int)

print('Operational threshold:', round(threshold, 3))
print('Precision:', round(precision_score(y_test, pred_operational, zero_division=0), 3))
print('Recall:', round(recall_score(y_test, pred_operational, zero_division=0), 3))
print('F1:', round(f1_score(y_test, pred_operational, zero_division=0), 3))

if hasattr(selected_model.named_steps['model'], 'feature_importances_'):
    feat_names = selected_model.named_steps['prep'].get_feature_names_out()
    importances = selected_model.named_steps['model'].feature_importances_
    fi = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False)
    display(fi.head(20))
else:
    print('Selected model has no native tree importances; use coefficients or permutation importance.')

## 4) Evaluation & Interpretation

Use both statistical and operational interpretation:
- Report `average_precision` and F1 for out-of-sample predictive quality.
- Choose a threshold aligned to operations (e.g., target high recall for `Transferred`).
- Convert model outputs into risk tiers for case-management action.

**Business interpretation guide**
- A high-recall setting reduces missed at-risk residents (fewer false negatives), but increases false positives and staff workload.
- A high-precision setting reduces unnecessary interventions, but risks missing vulnerable residents.

In this context, false negatives are typically more costly because a missed high-risk resident may receive delayed support.

In [ ]:
# Confusion matrix + risk tier outputs for operations
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_operational, ax=ax)
ax.set_title(f'Confusion matrix @ threshold={threshold:.2f}')
plt.show()

print('ROC-AUC:', round(roc_auc_score(y_test, proba), 3))
print('PR-AUC (avg precision):', round(average_precision_score(y_test, proba), 3))

scored = X_test.copy()
scored['actual_transferred'] = y_test.values
scored['pred_transfer_prob'] = proba
scored['risk_tier'] = pd.cut(
    scored['pred_transfer_prob'],
    bins=[-0.001, threshold, 0.75, 1.0],
    labels=['Monitor', 'Medium', 'High']
)

display(scored[['pred_transfer_prob', 'risk_tier', 'actual_transferred']].sort_values('pred_transfer_prob', ascending=False).head(15))

scored[['pred_transfer_prob', 'risk_tier', 'actual_transferred']].to_csv(
    ARTIFACT_DIR / 'resident_transfer_risk_scored_sample.csv', index=False
)
print('Saved:', ARTIFACT_DIR / 'resident_transfer_risk_scored_sample.csv')

In [ ]:
# Persist artifacts for reproducibility and deployment handoff
import joblib

joined.to_csv(ARTIFACT_DIR / 'resident_transfer_joined_dataset.csv', index=False)
joblib.dump(selected_model, ARTIFACT_DIR / 'resident_transfer_risk_model.joblib')

metrics_payload = {
    'selected_model': best_name,
    'threshold': float(threshold),
    'roc_auc': float(roc_auc_score(y_test, proba)),
    'avg_precision': float(average_precision_score(y_test, proba)),
    'precision_at_threshold': float(precision_score(y_test, pred_operational, zero_division=0)),
    'recall_at_threshold': float(recall_score(y_test, pred_operational, zero_division=0)),
    'f1_at_threshold': float(f1_score(y_test, pred_operational, zero_division=0)),
}

pd.DataFrame([metrics_payload]).to_csv(ARTIFACT_DIR / 'resident_transfer_risk_metrics.csv', index=False)
print('Saved artifacts to:', ARTIFACT_DIR)

## 5) Causal and Relationship Analysis

### What relationships appear strongest?
Interpret the top model features in three groups:
1. **Incident burden** (count, severity, unresolved high-severity incidents)
2. **Education momentum** (attendance/progress means and deltas)
3. **Case context** (initial risk, referral source, reintegration status)

### Do these relationships make theoretical sense?
Yes, generally:
- Higher unresolved/high-severity incidents can indicate instability and transfer pressure.
- Declining attendance/progress can signal broader disengagement and case complexity.
- Reintegration delays can coincide with prolonged or difficult case pathways.

### Correlation vs causation
This pipeline is primarily **predictive**. Feature importance and coefficients describe association structure, not proof of causal effect.

Causal claims are limited because:
- No random assignment of interventions.
- Possible confounding (safehouse practices, social worker assignment, referral channel).
- Some features may be downstream proxies for latent case severity.

So the defensible claim is: these variables help predict transfer risk; they do not independently prove transfer causes.

### Practical implication despite non-causal limits
Even without strict causal identification, stable predictive patterns are operationally useful for early triage and targeted support, as long as decisions include human review.

## 6) Deployment Notes

### Current integration pattern in this repo
The service already exposes analytics endpoints and loads notebook/model artifacts from `ml-pipelines/artifacts`:
- `ml-service/app/main.py` (e.g., `/donations/analytics`, `/reports/tier1-analytics`)

### Suggested transfer-risk integration
1. Save model and preprocessing pipeline artifact from this notebook (e.g., `resident_transfer_risk_model.joblib`).
2. Add an API endpoint like `/residents/transfer-risk` in `ml-service/app/main.py`.
3. Endpoint loads current resident features, applies model, returns probability and risk tier.
4. Frontend dashboard consumes risk tiers and shows recommended intervention playbooks.

### Example API response shape
```json
{
  "residentId": 123,
  "predTransferProb": 0.81,
  "riskTier": "High",
  "topDrivers": ["unresolved_high_count", "incident_severity_mean", "progress_delta"]
}
```

### Governance notes
- Retrain monthly or on drift triggers (case mix / referral mix changes).
- Track subgroup metrics and threshold impact on workload.
- Keep a human-in-the-loop rule for high-stakes case decisions.